# Paid-safe preflight

This notebook contains no extraction loop. It inspects configuration, local official data, validated signatures, the token budget, credential fingerprints and the production-cache gate before training.

In [2]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print(ROOT)

/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1


In [3]:
from dataclasses import asdict
from us10y_fomc.config import ProjectPaths, load_project_config

config = load_project_config(ROOT)
paths = ProjectPaths(ROOT).ensure()
display(asdict(config.runtime))
assert not config.runtime.run_paid_extraction or config.runtime.confirm_paid_extraction

{'download_fomc_minutes': False,
 'run_openai_preflight': False,
 'confirm_api_key': False,
 'run_paid_extraction': False,
 'confirm_paid_extraction': False,
 'retry_quarantined_pairs': False,
 'confirm_quarantine_retry': False,
 'max_new_pairs': 0,
 'train_price_benchmark': False,
 'train_historical_backbone': False,
 'train_fusion_models': False,
 'run_walk_forward': False}

In [4]:
completed = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'run_preflight.py')],
    cwd=ROOT, check=True, text=True
)
report = json.loads((paths.audit_reports / 'preflight_report.json').read_text())
display(report)

{
  "treasury_panel": true,
  "document_integrity": true,
  "fred_rate_audit": true,
  "token_budget": true,
  "openai_model_access": true,
  "production_extraction_complete": true,
  "missing_pairs": 0,
  "fred_key_fingerprint": "a07b7ee6555a",
  "openai_key_fingerprint": null,
  "safe_paid_switches": true,
  "go_for_training": true
}


{'treasury_panel': True,
 'document_integrity': True,
 'fred_rate_audit': True,
 'token_budget': True,
 'openai_model_access': True,
 'production_extraction_complete': True,
 'missing_pairs': 0,
 'fred_key_fingerprint': 'a07b7ee6555a',
 'openai_key_fingerprint': None,
 'safe_paid_switches': True,
 'go_for_training': True}

Training may start only when `go_for_training` is true. A false `production_extraction_complete` is expected before the paid extraction and is a hard stop, not a notebook defect.